In [2]:
import pygad
import numpy as np
import matplotlib.pyplot as plt
import time
from fitness import convertir_cromosoma, calcular_fitness, contador_molinos

### Inicialización del AG

In [3]:
def _construir_ga(poblacion, mutacion):
    
    def _fitness_pygad(ga_instance, cromosoma, idx):
        return calcular_fitness(convertir_cromosoma(cromosoma))
    
    # Función para tener los datos de cada generación
    def on_generation_interna(ga):
        # Si es la primera generación, creamos las listas dentro del objeto ga
        if not hasattr(ga, "hist_mejor"):
            ga.hist_mejor = []
            ga.hist_promedio = []
            ga.hist_std = []
            
        # Agregamos los datos
        fits = ga.last_generation_fitness
        ga.hist_mejor.append(float(np.max(fits)))
        ga.hist_promedio.append(float(np.mean(fits)))
        ga.hist_std.append(float(np.std(fits)))

    return pygad.GA(
        sol_per_pop          = poblacion,
        num_genes            = 50,
        gene_type            = int,
        gene_space           = range(0, 20),
        fitness_func         = _fitness_pygad,
        parent_selection_type= "tournament",
        K_tournament         = 3,
        keep_elitism         = 5,
        crossover_type       = "two_points",
        num_parents_mating   = poblacion//2,
        mutation_type        = "random",
        mutation_probability = mutacion,
        num_generations      = 500,
        stop_criteria        = ["reach_53.25", "saturate_50"],
        on_generation        = on_generation_interna
    )


In [5]:
import pandas as pd
import numpy as np
import time

TARGET_FITNESS = 53.25  # El criterio "reach" del stop_criteria

instancias = [
    {"nombre": "Instancia 1 ('NORMAL')", "poblacion": 100, "mutacion": 0.05},
    {"nombre": "Instancia 2 (MUTACIÓN)", "poblacion": 100, "mutacion": 0.01},
    {"nombre": "Instancia 3 (POBLACIÓN)", "poblacion":  50, "mutacion": 0.05},
]

EJECUCIONES = 30

filas = []
for inst in instancias:
    print(f"Ejecutando {inst['nombre']}...")
    gens_list  = []
    tiempos    = []
    hits       = 0

    for _ in range(EJECUCIONES):
        ga = _construir_ga(inst["poblacion"], inst["mutacion"])

        start = time.time()
        ga.run()
        duracion = time.time() - start
        # campo0, fitness, campo2
        _, fitness, _ = ga.best_solution()

        gens_list.append(ga.generations_completed)
        tiempos.append(round(duracion, 2))

        # Cuenta como éxito si alcanzó el target
        if float(fitness) >= TARGET_FITNESS:
            hits += 1

    filas.append({
        "Instancia"         : inst["nombre"],
        "Hit Rate (%)"      : round(hits / EJECUCIONES * 100, 1),
        "Prom. Generaciones": round(np.mean(gens_list), 1),
        "Min. Generaciones" : int(np.min(gens_list)),
        "Prom. Tiempo (s)"  : round(np.mean(tiempos), 2),
    })

df = pd.DataFrame(filas).set_index("Instancia")
df


Ejecutando Instancia 1 ('NORMAL')...
Ejecutando Instancia 2 (MUTACIÓN)...
Ejecutando Instancia 3 (POBLACIÓN)...


,Hit Rate (%),Prom. Generaciones,Min. Generaciones,Prom. Tiempo (s)
Instancia,,,,
Instancia 1 ('NORMAL'),100.0,8.8,2,0.16
Instancia 2 (MUTACIÓN),100.0,6.7,1,0.11
Instancia 3 (POBLACIÓN),100.0,8.5,2,0.08
